# 01 Data Profiling

Profile Bronze, Silver, and Gold CMS DE-SynPUF tables after running ingestion and transforms.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

DB = Path('../data/processed/desynpuf.duckdb')
con = duckdb.connect(str(DB), read_only=True)
tables = con.execute("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()
tables

In [ ]:
row_counts = []
for table in tables['table_name']:
    rows = con.execute(f'SELECT COUNT(*) AS rows FROM "{table}"').fetchone()[0]
    row_counts.append({'table_name': table, 'rows': rows})
pd.DataFrame(row_counts).sort_values('table_name')

In [ ]:
con.execute("""
    SELECT year, COUNT(*) AS beneficiary_years,
           SUM(total_synthetic_cost) AS total_synthetic_cost,
           AVG(total_synthetic_cost) AS avg_synthetic_cost
    FROM gold_patient_year_summary
    GROUP BY year
    ORDER BY year
""").df()